In [ ]:
import pandas as pd

In [ ]:
!pip uninstall -y transformers accelerate
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate

Found existing installation: transformers 5.8.0.dev0
Uninstalling transformers-5.8.0.dev0:
  Successfully uninstalled transformers-5.8.0.dev0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-uq9_6qyd
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-uq9_6qyd
  Resolved https://github.com/huggingface/transformers.git to commit 28ab024e1060d3f7e41d6370850c5a7954c8a6f3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11819754 sha256=04a14ff5f24d6e392e38dc5e768088bae49ecf270b0518d414b2f2b4a23cb0cb
  Stored in directory: /tmp/pip-ephem-wheel-cache-eatlerpo/wheels/54/cb/3f/83103de5575c534436d6a468

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.0 MB/s eta 0:00:00


## Sample

In [ ]:
data = pd.read_csv("pararel.csv")

In [ ]:
data.head()

,subject,rel_lemma,object,rel_p_id,query
0,Newport County A.F.C.,is-headquarter,Newport,P159,Newport County A.F.C. is headquartered in
1,Norway,capital-city-of,Oslo,P1376,"Norway's capital city,"
2,WWE,is-headquarter,Stamford,P159,WWE is headquartered in
3,Princeton University Press,is-headquarter,Princeton,P159,Princeton University Press is headquartered in
4,Internet censorship,is-subclass,censorship,P279,Internet censorship is a subclass of


In [ ]:
data["rel_p_id"].value_counts()

,count
rel_p_id,
P37,1157
P176,639
P30,436
P279,388
P178,386
P1376,385
P27,344
P159,290
P140,277


In [ ]:
p37 = data[data["rel_p_id"]=="P37"].sample(n=250)

In [ ]:
p30 = data[data["rel_p_id"]=="P30"].sample(n=250)

In [ ]:
p1376 = data[data["rel_p_id"]=="P1376"].sample(n=250)

In [ ]:
p27 = data[data["rel_p_id"]=="P27"].sample(n=250)

In [ ]:
p279 = data[data["rel_p_id"]=="P279"].sample(n=250)

In [ ]:
select = pd.concat([p37, p30, p1376, p27, p279])

In [ ]:
select.shape

(1250, 5)

In [ ]:
select.to_csv("select_pararel.csv", index=False)

## Qwen

In [ ]:
pip install transformers accelerate torch torchvision pillow

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import torch.nn.functional as F

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
import transformers
transformers.utils.logging.disable_progress_bar()

In [ ]:
import json

In [ ]:
from transformers import logging
logging.set_verbosity_error()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    torch_dtype=torch.bfloat16,
    device_map="cuda")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [ ]:
def process_data(input_file, output_file, prompt='Complete the sentence. Answer with one word.'):
  df = pd.read_csv(input_file)
  if "prompt" not in df.columns:
    df["prompt"] = None
  if "model_output" not in df.columns:
    df["model_output"] = None
  if "output_probs" not in df.columns:
    df["output_probs"] = None


  for i in tqdm(df.index):
    if pd.notna(df.at[i, "model_output"]):  # пропуск готовых
      continue

    messages = [{
        "role": "user", "content": f'{prompt} "{df.at[i, "query"]}"'
    }]
    df.at[i, "prompt"] = messages[0]["content"]
    inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
      **inputs,
      max_new_tokens=40,
      return_dict_in_generate=True,
      output_scores=True,
      )

    # Декодирование ответа
    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = outputs.sequences[:, prompt_len:]
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    df.at[i, "model_output"] = output

    # Извлечение логитов / вероятностей
    probs_list = []
    for step, score in enumerate(outputs.scores):
        probs = F.softmax(score[0].float(), dim=-1)
        top_probs, top_ids = probs.topk(5)
        top_tokens = [
                tokenizer.decode([tid], skip_special_tokens=True)
                for tid in top_ids.tolist()
            ]
        pairs = [(tok, f"{p:.6f}") for tok, p in zip(top_tokens, top_probs.tolist())]
        probs_list.append(pairs)
    df.at[i, "output_probs"] = json.dumps(probs_list, ensure_ascii=False)

    if i % 100 == 0:  # периодическое сохранение
            df.to_csv(output_file, index=False)

  df.to_csv(output_file, index=False)
  return df

In [ ]:
process_data("select_pararel.csv", "neutral_res.csv")

  0%|          | 0/1250 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,prompt,model_output,output_probs
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,"Complete the sentence. Answer with one word. ""...",English,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,"Complete the sentence. Answer with one word. ""...",Russian,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,"Complete the sentence. Answer with one word. ""...",Dutch,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Symeon of Polotsk,mother-tongue,Russian,P37,The mother tongue of Symeon of Polotsk is,"Complete the sentence. Answer with one word. ""...",Belarusian,"[[[""Bel"", ""0.694391""], [""Old"", ""0.166411""], [""..."
4,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,"Complete the sentence. Answer with one word. ""...",Finnish,"[[[""F"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
...,...,...,...,...,...,...,...,...
1245,Latin poetry,is-subclass,poetry,P279,"Latin poetry, a type of","Complete the sentence. Answer with one word. ""...",verse,"[[[""ep"", ""0.549762""], [""Latin"", ""0.225119""], [..."
1246,hamate bone,is-subclass,bone,P279,"hamate bone, a type of","Complete the sentence. Answer with one word. ""...",bone,"[[[""car"", ""0.630816""], [""bone"", ""0.369184""], [..."
1247,pound cake,is-subclass,cake,P279,"pound cake, a type of","Complete the sentence. Answer with one word. ""...",cake,"[[[""cake"", ""1.000000""], [""#"", ""0.000000""], [""!..."
1248,criminal defense lawyer,is-subclass,lawyer,P279,"criminal defense lawyer, a type of","Complete the sentence. Answer with one word. ""...",professional,"[[[""professional"", ""0.437651""], [""profession"",..."


### Check

In [ ]:
neutral_res = pd.read_csv("neutral_res_qwen.csv")

In [ ]:
(neutral_res["object"] == neutral_res["model_output"]).sum()

np.int64(971)

In [ ]:
print(f"P37: {(neutral_res[neutral_res["rel_p_id"]=="P37"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P30: {(neutral_res[neutral_res["rel_p_id"]=="P30"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P1376: {(neutral_res[neutral_res["rel_p_id"]=="P1376"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P27: {(neutral_res[neutral_res["rel_p_id"]=="P27"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P279: {(neutral_res[neutral_res["rel_p_id"]=="P279"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")

P37: 228
P30: 231
P1376: 227
P27: 214
P279: 183


In [ ]:
filtered_qwen = neutral_res[
    (neutral_res.apply(lambda x: str(x["object"]).lower() in str(x["model_output"]).lower(), axis=1)) &
    (neutral_res["rel_p_id"] != "P279")
]

In [ ]:
filtered_qwen.shape

(900, 8)

In [ ]:
filtered_qwen.to_csv("filtered_qwen.csv", index=False)

In [ ]:
data = pd.read_csv("filtered_qwen.csv")

In [ ]:
data["consistent_sent"] = data["query"] + " " + data["object"]
data["conflict_sent"] = None

In [ ]:
import random

In [ ]:
data["sub_object"] = None
for i in data.index:
  id = data.at[i, "rel_p_id"]
  sub_options = list(set(data[data["rel_p_id"] == id]["object"].unique().tolist()) - set([data.at[i, "object"]]))
  sub_object = random.choice(sub_options)
  data.at[i, "sub_object"] = sub_object
  data.at[i, "conflict_sent"] = data.at[i, "query"] + " " + sub_object


In [ ]:
data = data.drop(["prompt", "model_output", "output_probs"], axis=1)

In [ ]:
data.head()

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch


In [ ]:
data.to_csv("qwen_context_sents.csv", index=False)

In [ ]:
def process_data(input_file, output_file, exp_n=1, prompt="neutral", confidence="percents"):

  df = pd.read_csv(input_file)
  if "prompt" not in df.columns:
    df["prompt"] = None
  if f"model_output_{exp_n}" not in df.columns:
    df[f"model_output_{exp_n}"] = None
  if f"output_probs_{exp_n}" not in df.columns:
    df[f"output_probs_{exp_n}"] = None

  for i in tqdm(df.index):
    if prompt == "neutral":
      prompt_text = "Complete the sentence. Answer with one word."
    elif prompt == "consistent":
      prompt_text = df.at[i, "consistent_sent"] + ". Complete the sentence. "
    elif prompt == "conflict":
      prompt_text = df.at[i, "conflict_sent"] + ". Complete the sentence. "
    elif prompt == "conflict_confident_user":
      prompt_text = "I believe, that " + df.at[i, "conflict_sent"] + ". Complete the sentence. "

    if confidence == "percents":
      prompt_text += """In addition, estimate, how confident are you in given answer from 0 to 100, where 0 is absolutely not confident and 100 is absolutely confident.
      Return only two items: the answer (sentence completion) and number (confidence estimation) separated by space without any other formatting. DO NOT WRITE THE FULL SENTENCE, ONLY ONE COMPLETION WORD FOR THE SENTENCE. Example - Sentence: The dog belongs to the family of Answer: mammals 100. """
    elif confidence == "verbalized":
      prompt_text += """In addition, estimate, how confident are you in given answer.
      Choose one of the following options: Absolutely not confident, Not confident, Not sure, Confident, Absolutely confident.
      Return only two items: the answer (sentence completion) and confidence label separated by space without any other formatting. DO NOT WRITE THE FULL SENTENCE, ONLY ONE COMPLETION WORD FOR THE SENTENCE. Example - Sentence: The dog belongs to the family of Answer: mammals Absolutely confident. """

    prompt_text += f"Your sentence: {df.at[i, 'query']}"

    if pd.notna(df.at[i, f"model_output_{exp_n}"]):  # пропуск готовых
      continue

    messages = [{
        "role": "user", "content": prompt_text
    }]
    df.at[i, "prompt"] = prompt_text
    inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
      **inputs,
      max_new_tokens=40,
      return_dict_in_generate=True,
      output_scores=True,
      )

    # Декодирование ответа
    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = outputs.sequences[:, prompt_len:]
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    df.at[i, f"model_output_{exp_n}"] = output

    # Извлечение логитов / вероятностей
    probs_list = []
    for step, score in enumerate(outputs.scores):
        probs = F.softmax(score[0].float(), dim=-1)
        top_probs, top_ids = probs.topk(5)
        top_tokens = [
                tokenizer.decode([tid], skip_special_tokens=True)
                for tid in top_ids.tolist()
            ]
        pairs = [(tok, f"{p:.6f}") for tok, p in zip(top_tokens, top_probs.tolist())]
        probs_list.append(pairs)
    df.at[i, f"output_probs_{exp_n}"] = json.dumps(probs_list, ensure_ascii=False)

    if i % 100 == 0:  # периодическое сохранение
            df.to_csv(output_file, index=False)

  df.to_csv(output_file, index=False)
  return df

In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_consistent_percents.csv", 1, "consistent", "percents")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,The official language of Hawaii is English. Co...,English 100,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,The mother tongue of Oleg Skripochka is Russia...,Russian 100,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,The official language of Flemish Community is ...,Dutch 100,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,The official language of Nousiainen is Finnish...,Finnish 100,"[[[""F"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,The mother tongue of Dmitry Merezhkovsky is Ru...,Russian 100,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Gustave Flourens is a citizen of France. Compl...,France 100,"[[[""France"", ""1.000000""], [""#"", ""0.000000""], [..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Poonam Sinha has a citizenship of India. Compl...,India 100,"[[[""India"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Bibi Andersson is a citizen of Sweden. Complet...,Sweden 100,"[[[""Sweden"", ""1.000000""], [""#"", ""0.000000""], [..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Priyanka Vadra has a citizenship of India. Com...,India 100,"[[[""India"", ""1.000000""], [""#"", ""0.000000""], [""..."


In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_consistent_verbalized.csv", 1, "consistent", "verbalized")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,The official language of Hawaii is English. Co...,English Confident,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,The mother tongue of Oleg Skripochka is Russia...,Russian Confident,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,The official language of Flemish Community is ...,Dutch Confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,The official language of Nousiainen is Finnish...,Finnish Absolutely confident,"[[[""F"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,The mother tongue of Dmitry Merezhkovsky is Ru...,Russian Confident,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Gustave Flourens is a citizen of France. Compl...,France Confident,"[[[""France"", ""1.000000""], [""#"", ""0.000000""], [..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Poonam Sinha has a citizenship of India. Compl...,India Confident,"[[[""India"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Bibi Andersson is a citizen of Sweden. Complet...,Sweden Confident,"[[[""Sweden"", ""1.000000""], [""#"", ""0.000000""], [..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Priyanka Vadra has a citizenship of India. Com...,India Confident,"[[[""India"", ""1.000000""], [""#"", ""0.000000""], [""..."


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_conflict_percents.csv", 1, "conflict", "percents")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,The official language of Hawaii is Swedish. Co...,Swedish 0,"[[[""Sw"", ""1.000000""], [""#"", ""0.000000""], [""!"",..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,The mother tongue of Oleg Skripochka is French...,French 100,"[[[""French"", ""1.000000""], [""#"", ""0.000000""], [..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,The official language of Flemish Community is ...,Dutch 50,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,The official language of Nousiainen is Indones...,Indonesian 0,"[[[""Ind"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,The mother tongue of Dmitry Merezhkovsky is Du...,Dutch 0,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Gustave Flourens is a citizen of Greece. Compl...,Greece 0,"[[[""G"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Poonam Sinha has a citizenship of Japan. Compl...,Japan 0,"[[[""Japan"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Bibi Andersson is a citizen of Norway. Complet...,Norway 100,"[[[""Nor"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Priyanka Vadra has a citizenship of Brazil. Co...,Brazil 0,"[[[""Brazil"", ""1.000000""], [""#"", ""0.000000""], [..."


In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_conflict_verbalized.csv", 1, "conflict", "verbalized")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,The official language of Hawaii is Swedish. Co...,English Not confident,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,The mother tongue of Oleg Skripochka is French...,French Confident,"[[[""French"", ""1.000000""], [""#"", ""0.000000""], [..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,The official language of Flemish Community is ...,Dutch Not confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,The official language of Nousiainen is Indones...,Indonesian Absolutely not confident,"[[[""Ind"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,The mother tongue of Dmitry Merezhkovsky is Du...,Dutch Not confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Gustave Flourens is a citizen of Greece. Compl...,Greece Not confident,"[[[""G"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Poonam Sinha has a citizenship of Japan. Compl...,Japan Absolutely confident,"[[[""Japan"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Bibi Andersson is a citizen of Norway. Complet...,Norway Confident,"[[[""Nor"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Priyanka Vadra has a citizenship of Brazil. Co...,Brazil Absolutely not confident,"[[[""Brazil"", ""1.000000""], [""#"", ""0.000000""], [..."


In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_neutral_percents.csv", 1, "neutral", "percents")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,Complete the sentence. Answer with one word.In...,English 95,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,Complete the sentence. Answer with one word.In...,Russian 95,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,Complete the sentence. Answer with one word.In...,Dutch 100,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,Complete the sentence. Answer with one word.In...,Finnish 100,"[[[""F"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,Complete the sentence. Answer with one word.In...,Russian 95,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Complete the sentence. Answer with one word.In...,France 95,"[[[""France"", ""1.000000""], [""#"", ""0.000000""], [..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Complete the sentence. Answer with one word.In...,Answer: Indian 95,"[[[""Answer"", ""0.588349""], [""India"", ""0.411651""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Complete the sentence. Answer with one word.In...,Sweden 95,"[[[""Sweden"", ""1.000000""], [""#"", ""0.000000""], [..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Complete the sentence. Answer with one word.In...,Indian 95,"[[[""Answer"", ""0.671348""], [""Indian"", ""0.328652..."


In [ ]:
process_data("qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_neutral_verbalized.csv", 1, "neutral", "verbalized")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,Complete the sentence. Answer with one word.In...,English Confident,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,Complete the sentence. Answer with one word.In...,Russian Confident,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,Complete the sentence. Answer with one word.In...,Dutch Confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,Complete the sentence. Answer with one word.In...,Finnish Confident,"[[[""F"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,Complete the sentence. Answer with one word.In...,Russian Confident,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,Complete the sentence. Answer with one word.In...,France Confident,"[[[""France"", ""1.000000""], [""#"", ""0.000000""], [..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,Complete the sentence. Answer with one word.In...,Indian Confident,"[[[""Indian"", ""1.000000""], [""#"", ""0.000000""], [..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,Complete the sentence. Answer with one word.In...,Sweden Confident,"[[[""Sweden"", ""1.000000""], [""#"", ""0.000000""], [..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,Complete the sentence. Answer with one word.In...,Indian Confident,"[[[""Indian"", ""1.000000""], [""#"", ""0.000000""], [..."


In [ ]:
process_data("/content/drive/MyDrive/диплом/qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_conflict_confident_user_percents.csv", 1, "conflict_confident_user", "percents")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,"I believe, that The official language of Hawai...",English 0,"[[[""English"", ""1.000000""], [""#"", ""0.000000""], ..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,"I believe, that The mother tongue of Oleg Skri...",French 20,"[[[""French"", ""1.000000""], [""#"", ""0.000000""], [..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,"I believe, that The official language of Flemi...",Dutch 95,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,"I believe, that The official language of Nousi...",Indonesian 0,"[[[""Ind"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,"I believe, that The mother tongue of Dmitry Me...",Russian 5,"[[[""Russian"", ""1.000000""], [""#"", ""0.000000""], ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,"I believe, that Gustave Flourens is a citizen ...",Greece 0,"[[[""G"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,"I believe, that Poonam Sinha has a citizenship...",Japan 0,"[[[""Japan"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,"I believe, that Bibi Andersson is a citizen of...",Norway 95,"[[[""Nor"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,"I believe, that Priyanka Vadra has a citizensh...",Brazil 0,"[[[""Brazil"", ""1.000000""], [""#"", ""0.000000""], [..."


In [ ]:
process_data("/content/drive/MyDrive/диплом/qwen_context_sents.csv", "/content/drive/MyDrive/диплом/qwen/qwen_conflict_confident_user_verbalized.csv", 1, "conflict_confident_user", "verbalized")

  0%|          | 0/900 [00:00<?, ?it/s]

,subject,rel_lemma,object,rel_p_id,query,consistent_sent,conflict_sent,sub_object,prompt,model_output_1,output_probs_1
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,The official language of Hawaii is English,The official language of Hawaii is Swedish,Swedish,"I believe, that The official language of Hawai...",Swedish False Not confident,"[[[""Sw"", ""1.000000""], [""#"", ""0.000000""], [""!"",..."
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,The mother tongue of Oleg Skripochka is Russian,The mother tongue of Oleg Skripochka is French,French,"I believe, that The mother tongue of Oleg Skri...",French Confident,"[[[""French"", ""1.000000""], [""#"", ""0.000000""], [..."
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,The official language of Flemish Community is ...,The official language of Flemish Community is ...,French,"I believe, that The official language of Flemi...",Dutch Not confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
3,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,The official language of Nousiainen is Finnish,The official language of Nousiainen is Indonesian,Indonesian,"I believe, that The official language of Nousi...",Indonesian Not confident,"[[[""Ind"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
4,Dmitry Merezhkovsky,mother-tongue,Russian,P37,The mother tongue of Dmitry Merezhkovsky is,The mother tongue of Dmitry Merezhkovsky is Ru...,The mother tongue of Dmitry Merezhkovsky is Dutch,Dutch,"I believe, that The mother tongue of Dmitry Me...",Dutch Not confident,"[[[""D"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
...,...,...,...,...,...,...,...,...,...,...,...
895,Gustave Flourens,is-citizen-of,France,P27,Gustave Flourens is a citizen of,Gustave Flourens is a citizen of France,Gustave Flourens is a citizen of Greece,Greece,"I believe, that Gustave Flourens is a citizen ...",Greece Not confident,"[[[""G"", ""1.000000""], [""#"", ""0.000000""], [""!"", ..."
896,Poonam Sinha,have-citizenship-of,India,P27,Poonam Sinha has a citizenship of,Poonam Sinha has a citizenship of India,Poonam Sinha has a citizenship of Japan,Japan,"I believe, that Poonam Sinha has a citizenship...",Japan Not confident,"[[[""Japan"", ""1.000000""], [""#"", ""0.000000""], [""..."
897,Bibi Andersson,is-citizen-of,Sweden,P27,Bibi Andersson is a citizen of,Bibi Andersson is a citizen of Sweden,Bibi Andersson is a citizen of Norway,Norway,"I believe, that Bibi Andersson is a citizen of...",Norway Confident,"[[[""Nor"", ""1.000000""], [""#"", ""0.000000""], [""!""..."
898,Priyanka Vadra,have-citizenship-of,India,P27,Priyanka Vadra has a citizenship of,Priyanka Vadra has a citizenship of India,Priyanka Vadra has a citizenship of Brazil,Brazil,"I believe, that Priyanka Vadra has a citizensh...",Brazil Absolutely not confident,"[[[""Brazil"", ""1.000000""], [""#"", ""0.000000""], [..."
